In [1]:
import arcpy

In [2]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input domain raster "rdomain_GBC15a_bathy_p26920_v6642" with desired extent of output boundary polygon
rdomain_gbc = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\rdomain_GBC15a_bathy_p26920_v6642"

# Output boundary polygon defining study area extent
boundary = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\MASTER_THESIS.gdb\boundary_thesis_p26920"

In [3]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [ ]:
# ── 1. CREATE RECTANGULAR POLYGON (SMOOTH STRAIGHT EDGES, NO GAPS) FROM RASTER DOMAIN ────────────────

# set environment coordinate system: horizontal EPSG:26920 - removes vertical coordinate system (VIVD09 height) present in input raster domain polygon so output boundary polygon has only projected (and corresponding geographic)coordinate system, since this polygon is a flat boundary outline (NOT a direct representation of raster data like the raster domain polygon is)
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(26920)

# Minimum Bounding Geometry (Data Management Tools):
arcpy.management.MinimumBoundingGeometry(
    in_features=rdomain_gbc,
    out_feature_class=boundary,
    # 'envelope' creates perfectly rectangular box with sides aligned to N/S axes (removes chance of sides being diagonal/misaligned to map)
    geometry_type="ENVELOPE",
    # outputs single polygon encompassing all input features (like non-continuous polygons from land gaps in raster/etc.)
    group_option="ALL"
)

# *** reset coordinate system *** so it does NOT affect Calculate Geometry Attributes tool's manually set coordinate system parameter in the block below and/or cause any chance of reprojecting boundary polygon on the fly 
arcpy.ClearEnvironment("outputCoordinateSystem")

In [4]:
# ── 2. ADD COLUMNS TO ATTRIBUTE TABLE WITH MIN/MAX LON AND LAT ───────────────────────────────────────

# Calculate Geometry Attributes (Data Management Tools):
arcpy.management.CalculateGeometryAttributes(
    in_features=boundary,
    geometry_property=[
    #   ["field name",          "property"]
        ["x_coordinate_min_dd", "EXTENT_MIN_X"],
        ["x_coordinate_max_dd", "EXTENT_MAX_X"],
        ["y_coordinate_min_dd", "EXTENT_MIN_Y"],
        ["y_coordinate_max_dd", "EXTENT_MAX_Y"],
        ["area_geodesic_m",     "AREA_GEODESIC"]
    ],
    # calculate extent coordinates in decimal degrees using geographic coordinate system (EPSG:4269) that projected coordinate system of input polygon (EPSG:26920) is already based on * same datum (NAD 83) for both pcs and gcs = no reprojection *
    coordinate_system=arcpy.SpatialReference(4269),
    # geodesic area returned in square meters
    area_unit="SQUARE_METERS"
    # leave 'coordinate_format' as default ("same as input") - coordinate system parameter set to * geographic * coordinate system = outputs of extent properties will already be in decimal degrees
)

<Result 'C:\\Users\\kkeii\\OneDrive\\Documents\\ArcGIS\\Files\\THESIS_FILES\\Geodatabases\\Thesis_geodatabases\\MASTER_THESIS.gdb\\boundary_thesis_p26920'>